In [ ]:
# run this first, then pick the 5 csv files from the Data folder when the box pops up
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
raw = pd.read_csv('bay_area_modeling_table.csv')
dash = pd.read_csv('dashboard_data.csv')
eth = pd.read_csv('uc_admissions_summary_by_ethnicity.csv')
disc = pd.read_csv('uc_freshman_admission_by_discipline.csv')
trmaj = pd.read_csv('uc_transfer_admission_by_major.csv')

In [ ]:
# q1: average number of campuses an applicant applied to in 2025
y = raw[raw['fall_term'] == 2025]
big = y[y['campus'] != 'Universitywide']['applicants'].sum()
one = y[y['campus'] == 'Universitywide']['applicants'].sum()
print(round(big / one, 2))

In [ ]:
# q2: ucla 2025 admit rate for ca public high school applicants
s = dash[(dash['fall_term'] == 2025) & (dash['campus'] == 'Los Angeles') & (dash['school_type'] == 'High Schools (Public)')]
print(round(s['admits'].sum() * 100 / s['applicants'].sum(), 2))

In [ ]:
# q3: campus where cs hurts the admit rate the most vs its own overall rate
d25 = disc[disc['fall_term'] == 2025]
ov = d25[d25['broad_discipline'] == 'All disciplines'].set_index('campus')['admit_rate']
cs = d25[d25['broad_discipline'] == 'Computer Science'].set_index('campus')['admit_rate']
pen = (cs - ov).dropna()
print(pen.idxmin())

In [ ]:
# q4: iqr of berkeley cs admit gpa in 2025 (75th percentile minus 25th percentile)
b = trmaj[(trmaj['campus'] == 'Berkeley') & (trmaj['broad_discipline'] == 'Computer Science') & (trmaj['major'] == 'ComputerScience')]
print(round(b['admit_gpa_p75'].iloc[0] - b['admit_gpa_p25'].iloc[0], 2))

In [ ]:
# q5: how many of 9 campuses had white admit rate higher than hispanic/latino(a) rate in 2025
e25 = eth[(eth['fall_term'] == 2025) & (eth['campus'] != 'Systemwide')]
ap = e25[e25['count_type'] == 'App'].pivot_table(index='campus', columns='ethnicity', values='n', aggfunc='sum')
ad = e25[e25['count_type'] == 'Adm'].pivot_table(index='campus', columns='ethnicity', values='n', aggfunc='sum')
white_rate = ad['White'] / ap['White']
hispanic_rate = ad['Hispanic/Latino(a)'] / ap['Hispanic/Latino(a)']
print(int((white_rate > hispanic_rate).sum()))

In [ ]:
# q6: systemwide 2025, white or hispanic/latino(a) higher admit rate
sysw = eth[(eth['fall_term'] == 2025) & (eth['campus'] == 'Systemwide')]
wr = sysw[(sysw['ethnicity'] == 'White') & (sysw['count_type'] == 'Adm')]['n'].iloc[0] / sysw[(sysw['ethnicity'] == 'White') & (sysw['count_type'] == 'App')]['n'].iloc[0]
hr = sysw[(sysw['ethnicity'] == 'Hispanic/Latino(a)') & (sysw['count_type'] == 'Adm')]['n'].iloc[0] / sysw[(sysw['ethnicity'] == 'Hispanic/Latino(a)') & (sysw['count_type'] == 'App')]['n'].iloc[0]
print('Hispanic/Latino(a)' if hr > wr else 'White')

In [ ]:
# q7: bay area 2023 grads who enrolled in a ca community college within 12 months
bay = ['Alameda','Contra Costa','Marin','Napa','San Francisco','San Mateo','Santa Clara','Solano','Sonoma']
sub = raw[(raw['fall_term'] == 2023) & (raw['campus'] == 'Universitywide') & (raw['county'].isin(bay))]
print(round(sub['enrolled_ccc'].sum() * 100 / sub['graduates'].sum(), 2))

In [ ]:
# q8: mission san jose 2023, share of a-g completers who applied to uc
m = raw[(raw['high_school'] == 'MISSION SAN JOSE HIGH SCHOOL') & (raw['fall_term'] == 2023) & (raw['campus'] == 'Universitywide')]
print(round(m['applicants'].iloc[0] * 100 / m['ag_completers'].iloc[0], 2))

In [ ]:
# q9: distinct ca public high schools with at least one uc applicant in 2025
n = raw[(raw['fall_term'] == 2025) & (raw['campus'] == 'Universitywide') & (raw['applicants'] > 0) & (raw['school_type'] == 'High Schools (Public)')]['high_school'].nunique()
print(n)

In [ ]:
# q10: of the five schools, which beat its expected berkeley admit rate most 2022-2025
five = ['HERCULES HIGH SCHOOL','MISSION SENIOR HIGH SCHOOL','MONTEREY TRAIL HIGH SCHOOL','PHILLIP & SALA BURTON ACAD HS','RANCHO SAN JUAN HIGH SCHOOL']
b = dash[(dash['campus'] == 'Berkeley') & (dash['fall_term'].between(2022, 2025)) & (dash['high_school'].isin(five))]
print(b.groupby('high_school')['admit_rate_residual'].mean().idxmax())